In [6]:
import xarray as xr
import numpy as np
import pandas as pd
from ERA5_functions import *

In [7]:
ds = xr.open_mfdataset('/data/keeling/a/rytam2/a/iema_output/variables_202510210102.nc')

In [13]:
# Extract vars from dataset - raw var 
t2m = ds['t2m']
dp = ds['dp']

#tier 1 vars - dependent only on raw vars 
rh = rel_hum(dp, t2m)
#tier2
T_w = wbt(rh, t2m)
# tier 3 vars - dependent on raw/t1/t2 vars 
T_wg = k_to_f(wbgt(t2m, T_w))


bulk model, sectional model, modal model, particle resolved model

##### WBGT

In [19]:
fpath = '/data/keeling/a/rytam2/a/iema_output/arcgis_toprocess/'
##### functions
def yrly_days_exceed_thres(ds_hrly,thres):    
    # Resample hourly data to daily, check if any True in each day
    daily_exceeds = (ds_hrly > thres).resample(time='1D').max()    
    # Convert boolean daily flags to int (1 if exceeded threshold any hour that day)
    daily_exceeds_int = daily_exceeds.astype(int)
    # Group by year and sum to get count of days per year
    count_days_per_year = daily_exceeds_int.groupby('time.year').sum(dim='time').rename('YEARLY_DAYS_OVER_'+str(thres))
    jja_days_per_year = daily_exceeds_int.sel(time=daily_exceeds_int['time.month'].isin([6, 7, 8])).groupby('time.year').sum(dim='time').rename('YEARLY_JJADAYS_OVER_'+str(thres))
    return count_days_per_year, jja_days_per_year
    
def yrly_hours_exceed_thres(ds_hrly,thres):    
    flag_counts_passed_thres = (ds_hrly > thres).astype(int)
    count_hrs_per_year = flag_counts_passed_thres.groupby('time.year').sum(dim='time').rename('YEARLY_HOURS_OVER_'+str(thres))
    jja_hrs_per_year = flag_counts_passed_thres.sel(time=flag_counts_passed_thres['time.month'].isin([6, 7, 8])).groupby('time.year').sum(dim='time').rename('YEARLY_JJADAYS_OVER_'+str(thres))
    return count_hrs_per_year, jja_hrs_per_year

def daily_hours_exceed_thres(ds_hrly,thres):    
    flag_counts_passed_thres = (ds_hrly > thres).astype(int)
    count_hrs_per_year = flag_counts_passed_thres.resample(time='1D').sum(dim='time').rename('DAIY_HRS_OVER_'+str(thres))
    return count_hrs_per_year

In [51]:
## daily
daily_wbgt_mean = T_wg.resample(time='1D').mean() #F
daily_wbgt_max = T_wg.resample(time='1D').max() #F
daily_wbgt_min = T_wg.resample(time='1D').min() #F
daily_wbgt_hr_80 = daily_hours_exceed_thres(T_wg,80)
daily_wbgt_hr_85 = daily_hours_exceed_thres(T_wg,85)

daily_wbgt = xr.Dataset({'mean':daily_wbgt_mean, 'max':daily_wbgt_max, 'min':daily_wbgt_min, 'hr_over_80':daily_wbgt_hr_80, 'hr_over_85':daily_wbgt_hr_85})
daily_wbgt.to_netcdf(fpath+'daily_wbgt_by_county_2016-2024'+'.nc')

In [36]:
## yearly
yearly_wbgt_d_80,yearly_wbgt_jjad_80 = (yrly_days_exceed_thres(T_wg,80))
yearly_wbgt_d_85,yearly_wbgt_jjad_85 = (yrly_days_exceed_thres(T_wg,85))

yearly_wbgt_hr_80, yearly_wbgt_jjahr_80 = yrly_hours_exceed_thres(T_wg,80)
yearly_wbgt_hr_85, yearly_wbgt_jjahr_85 = yrly_hours_exceed_thres(T_wg,85)

yearly_absdailymax = daily_wbgt_max.groupby('time.year').max().rename('YEARLY_ABS_MAX')
# yearly_absdailymax_month = 
yearly_jjaavg_dailymax = daily_wbgt_max.sel(time=daily_wbgt_max['time.month'].isin([6, 7, 8])).groupby('time.year').mean(dim='time').rename('YEARLY_JJAAVG_DAILY_MAX')
yearly_jjaavg_dailymin = daily_wbgt_min.sel(time=daily_wbgt_min['time.month'].isin([6, 7, 8])).groupby('time.year').mean(dim='time').rename('YEARLY_JJAAVG_DAILY_MIN')

yearlyavg_junavg_wbgt = daily_wbgt_mean.sel(time=daily_wbgt_max['time.month'].isin([6])).groupby('time.year').mean(dim='time').rename('YEARLY_JUN_MEAN')
yearlyavg_julavg_wbgt = daily_wbgt_mean.sel(time=daily_wbgt_max['time.month'].isin([7])).groupby('time.year').mean(dim='time').rename('YEARLY_JUL_MEAN')
yearlyavg_augavg_wbgt = daily_wbgt_mean.sel(time=daily_wbgt_max['time.month'].isin([8])).groupby('time.year').mean(dim='time').rename('YEARLY_AUG_MEAN')


yearly_wbgt = xr.Dataset({
    yearly_wbgt_d_80.name: yearly_wbgt_d_80,
    yearly_wbgt_jjad_80.name: yearly_wbgt_jjad_80,
    yearly_wbgt_d_85.name: yearly_wbgt_d_85,
    yearly_wbgt_jjad_85.name:yearly_wbgt_jjad_85,
    yearly_wbgt_hr_80.name:yearly_wbgt_hr_80,
    yearly_wbgt_jjahr_80.name:yearly_wbgt_jjahr_80,
    yearly_wbgt_hr_85.name:yearly_wbgt_hr_85,
    yearly_wbgt_jjahr_85.name:yearly_wbgt_jjahr_85,
    yearly_absdailymax.name:yearly_absdailymax,
    yearly_jjaavg_dailymax.name:yearly_jjaavg_dailymax,
    yearly_jjaavg_dailymin.name:yearly_jjaavg_dailymin,
    yearlyavg_junavg_wbgt.name:yearlyavg_junavg_wbgt,
    yearlyavg_julavg_wbgt.name:yearlyavg_julavg_wbgt,
    yearlyavg_augavg_wbgt.name:yearlyavg_augavg_wbgt,
})

# yearly_wbgt.to_netcdf(fpath+'yearly_wbgt_by_county_2016-2024'+'.nc')

In [54]:
summary_wbgt.keys()

KeysView(<xarray.Dataset> Size: 122kB
Dimensions:                     (lat: 29, lon: 27)
Coordinates:
  * lat                         (lat) float32 116B 43.25 43.0 ... 36.5 36.25
  * lon                         (lon) float32 108B 267.2 267.5 ... 273.5 273.8
    county                      (lat, lon) <U11 34kB dask.array<chunksize=(29, 27), meta=np.ndarray>
Data variables: (12/16)
    TOTAL_DAYS_OVER_80          (lat, lon) int64 6kB dask.array<chunksize=(29, 27), meta=np.ndarray>
    YEARLY_AVG_DAYS_OVER_80     (lat, lon) float64 6kB dask.array<chunksize=(29, 27), meta=np.ndarray>
    TOTAL_JJADAYS_OVER_80       (lat, lon) int64 6kB dask.array<chunksize=(29, 27), meta=np.ndarray>
    YEARLY_JJAAVG_DAYS_OVER_80  (lat, lon) int64 6kB dask.array<chunksize=(29, 27), meta=np.ndarray>
    TOTAL_DAYS_OVER_85          (lat, lon) int64 6kB dask.array<chunksize=(29, 27), meta=np.ndarray>
    YEARLY_AVG_DAYS_OVER_85     (lat, lon) float64 6kB dask.array<chunksize=(29, 27), meta=np.ndarray>
    ...

In [38]:
## Summary 
# Avg Days/Hours WBGT above 80/85 in June/Jul/Aug across years; Max/Min WBGT across years - calculate
# WBGT JJA Daily Av. High	WBGT JJA Daily Av. Low - remove

decadetot_wbgt_d_80 = yearly_wbgt_d_80.sum(dim='year').rename('TOTAL_DAYS_OVER_80')
decadeavg_wbgt_d_80 = yearly_wbgt_d_80.mean(dim='year').rename('YEARLY_AVG_DAYS_OVER_80')
decadetot_wbgt_jjad_80 = yearly_wbgt_jjad_80.sum(dim='year').rename('TOTAL_JJADAYS_OVER_80')
decadeavg_wbgt_jjad_80 = yearly_wbgt_jjad_80.mean(dim='year').rename('YEARLY_JJAAVG_DAYS_OVER_80')

decadetot_wbgt_d_85 = yearly_wbgt_d_85.sum(dim='year').rename('TOTAL_DAYS_OVER_85')
decadeavg_wbgt_d_85 = yearly_wbgt_d_85.mean(dim='year').rename('YEARLY_AVG_DAYS_OVER_85')
decadetot_wbgt_jjad_85 = yearly_wbgt_jjad_85.sum(dim='year').rename('TOTAL_JJADAYS_OVER_85')
decadeavg_wbgt_jjad_85 = yearly_wbgt_jjad_85.mean(dim='year').rename('YEARLY_JJAAVG_DAYS_OVER_85')

decadetot_wbgt_hr_80 = yearly_wbgt_hr_80.sum(dim='year').rename('TOTAL_HRS_OVER_80')
decadetot_wbgt_jjahr_80 = yearly_wbgt_jjahr_80.sum(dim='year').rename('TOTAL_JJAHRS_OVER_80')
decadetot_wbgt_hr_85 = yearly_wbgt_hr_85.sum(dim='year').rename('TOTAL_HRS_OVER_85')
decadetot_wbgt_jjahr_85 = yearly_wbgt_jjahr_85.sum(dim='year').rename('TOTAL_JJAHRS_OVER_85')

decadeavg_junavg_wbgt = yearlyavg_junavg_wbgt.mean(dim='year').rename('DECADE_JUN_MEAN')
decadeavg_julavg_wbgt = yearlyavg_julavg_wbgt.mean(dim='year').rename('DECADE_JUL_MEAN')
decadeavg_augavg_wbgt = yearlyavg_augavg_wbgt.mean(dim='year').rename('DECADE_AUG_MEAN')

decade_absmax_wbgt = yearly_absdailymax.max(dim='year').rename('DECADE_RECORD_HIGH')

summary_wbgt = xr.Dataset({decadetot_wbgt_d_80.name:decadetot_wbgt_d_80,\
                           decadeavg_wbgt_d_80.name:decadeavg_wbgt_d_80,\
                           decadetot_wbgt_jjad_80.name:decadetot_wbgt_jjad_80,\
                           decadeavg_wbgt_jjad_80.name:decadetot_wbgt_jjad_80,\
                           decadetot_wbgt_d_85.name:decadetot_wbgt_d_85,\
                          decadeavg_wbgt_d_85.name:decadeavg_wbgt_d_85,\
                          decadetot_wbgt_jjad_85.name:decadetot_wbgt_jjad_85,\
                          decadeavg_wbgt_jjad_85.name:decadeavg_wbgt_jjad_85,\
                          decadetot_wbgt_hr_80.name:decadetot_wbgt_hr_80,\
                          decadetot_wbgt_jjahr_80.name:decadetot_wbgt_jjahr_80,\
                          decadetot_wbgt_hr_85.name:decadetot_wbgt_hr_85,\
                          decadetot_wbgt_jjahr_85.name:decadetot_wbgt_jjahr_85,\
                          decadeavg_junavg_wbgt.name:decadeavg_junavg_wbgt,\
                          decadeavg_julavg_wbgt.name:decadeavg_julavg_wbgt,\
                          decadeavg_augavg_wbgt.name:decadeavg_augavg_wbgt,\
                          decade_absmax_wbgt.name:decade_absmax_wbgt})

# summary_wbgt.to_netcdf(fpath+'summary_wbgt_by_county_2016-2024'+'.nc')

##### WBGT compiled from Maile's Code

Calculations are done, but files not saved: 
- yearly: merged_df
- summary: wbgt_tot

Realized in wbgt_tot that Maile's threshold for wbgt >90 is too high (all zeros) and therefore recalculated for 80/85. (See above)

coagulate;
aitken mode ==> accumulation mode

In [40]:
wbgt_daily = pd.read_csv('ERA5_WBGT_daily_2016_2025_ILCounties.csv')#,index_col =0) 
wbgt_tot = pd.read_csv('ERA5_WBGT_totalstats_table_2016_2025_ILCounties.csv')
wbgt_yearly = pd.read_csv('ERA5_WBGT_yearlystats_table_2016_2025_ILCounties.csv')
wbgt_daily['Day'] = pd.to_datetime(wbgt_daily['Day'])

In [50]:
wbgt_daily.keys()

Index(['Unnamed: 0', 'FIPS', 'mean', 'County', 'Day', 'max', 'min',
       'WBGT Hours above 85', 'WBGT Hours above 90'],
      dtype='object')

In [49]:
wbgt_daily['Day'] = pd.to_datetime(wbgt_daily['Day'])
# jja_h_85 = wbgt_daily[wbgt_daily['Day'].dt.month.([6, 7, 8])].groupby(['County',wbgt_daily['Day'].dt.year,wbgt_daily['Day'].dt.month])['WBGT Hours above 85'].sum().groupby(['County','Day']).sum().rename('WBGT JJA Total Hrs above 85').reset_index().rename(columns={'Day':'Year'})
jja_h_85 = wbgt_daily[wbgt_daily['Day'].dt.month.isin([6, 7, 8])].groupby(['County',wbgt_daily['Day'].dt.year])['WBGT Hours above 85'].sum().groupby(['County','Day']).sum().rename('WBGT JJA Total Hrs above 85').reset_index().rename(columns={'Day':'Year'})

# jja_h_90 = wbgt_daily[wbgt_daily['Day'].dt.month.([6, 7, 8])].groupby(['County',wbgt_daily['Day'].dt.year,wbgt_daily['Day'].dt.month])['WBGT Hours above 90'].sum().groupby(['County','Day']).sum().rename('WBGT JJA Total Hrs above 90').reset_index()
jja_h_90 = wbgt_daily[wbgt_daily['Day'].dt.month.isin([6, 7, 8])].groupby(['County',wbgt_daily['Day'].dt.year])['WBGT Hours above 90'].sum().groupby(['County','Day']).sum().rename('WBGT JJA Total Hrs above 90').reset_index().rename(columns={'Day':'Year'})

merged_df = pd.merge(wbgt_yearly, jja_h_85, on=['County', 'Year'], how='outer')
merged_df = pd.merge(merged_df, jja_h_90, on=['County', 'Year'], how='outer').drop('Unnamed: 0',axis=1).rename(columns={'WBGT Days above 85':'WBGT Year-total Days above 85',
                                                   'WBGT Hours above 85':'WBGT Year-total Hours above 85', 
                                                   'WBGT Days above 90':'WBGT Year-total  Days above 90' ,
                                                   'WBGT Hours above 90':'WBGT Year-total Days above 90',
                                                    'WBGT Av. 24h high JJA':'WBGT JJA-avg daily max',
                                                    'WBGT Av. 24h low JJA':'WBGT JJA-avg daily min'})

# merged_df.to_csv('yearlystats_hi_by_county_2016-2025.csv')

Index(['County', 'Year', 'FIPS', 'WBGT Year-total Days above 85',
       'WBGT Year-total Hours above 85', 'WBGT Year-total  Days above 90',
       'WBGT Year-total Days above 90', 'WBGT JJA-avg daily max',
       'WBGT JJA-avg daily min', 'WBGT JJA Total Hrs above 85',
       'WBGT JJA Total Hrs above 90'],
      dtype='object')

In [55]:
### Check if there are days in any location that passed WBGT threshold of 85/90 ==> leading to lowering threshold to 80/85 
## Extract year as a separate Series
year = wbgt_daily['Day'].dt.year
# Filter to summer months June, July, August
summer_data = wbgt_daily[wbgt_daily['Day'].dt.month.isin([6, 7, 8])]
# Group by 'County' and year, then count nonzero in 'WBGT Hours above 90'
count_nonzero = summer_data.groupby(['County', year])['WBGT Hours above 85'].apply(lambda x: (x != 0).sum())
count_nonzero.unique()

array([3, 2, 0, 1, 4])

In [46]:
## Avg Days/Hours WBGT above 90/104 in June/Jul/Aug across years; Max/Min WBGT across years - calculate
jja_h_85_avg = merged_df.groupby('County')['WBGT JJA Total Hrs above 85'].mean('year')
jja_h_85_avg = merged_df.groupby('County')['WBGT JJA Total Hrs above 90'].mean('year')

In [48]:
wbgt_tot.rename(columns={'WBGT Days above 85':'WBGT 2016-2025 Total Days above 85',
                         'WBGT Hours above 85':'WBGT 2016-2025 Total Hours above 85', 
                         'WBGT Days above 90':'WBGT 2016-2025 Total Days above 90' ,
                         'WBGT Hours above 90':'WBGT 2016-2025 Total Days above 90',
                         'WBGT JJA Daily Av. High':'WBGT 2016-2025-avg JJA-avg daily max',
                         'WBGT JJA Daily Av. Low':'WBGT 2016-2025-avg JJA-avg daily min'})

wbgt_tot.keys()

Index(['Unnamed: 0', 'County', 'FIPS', 'WBGT Days above 85',
       'WBGT Hours above 85', 'WBGT Days above 90', 'WBGT Hours above 90',
       'WBGT JJA Daily Av. High', 'WBGT JJA Daily Av. Low'],
      dtype='object')